# JSON documents — step by step

A hands-on lab. Run one cell at a time, read what happened, then move on. You will query JSONB specs with containment, path extraction, and GIN indexes.

Each step shows the **plain SQL** too, so you can copy it and run it in psql, the Lakebase SQL editor, or any tool. The Python just runs that SQL against your branch.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA` are set and the factory is loaded on your branch.

## Setup — connect to your branch

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import connect, run_sql

SCHEMA = os.environ.get('LB_SCHEMA', 'factory')
print('connected; schema:', SCHEMA)

## Step 1 — Inspect JSONB specs on parts

Each part has a `spec` column (JSONB). The contents vary: one part might have `{"load_capacity_kg": 500, "material": "steel"}`, another might have `{"tensile_strength_mpa": 1200, "coating": "zinc"}`. No schema migration needed when a new part type arrives with new fields. The SQL:

```sql
SELECT part_id, name, spec FROM factory.part LIMIT 3
```

In [ ]:
sql = "SELECT part_id, name, spec FROM factory.part LIMIT 3"
rows = run_sql(sql)
print("Sample part specs (JSONB, varying fields):")
for row in rows:
    print(f"  {row[0]}: {row[1]}")
    print(f"    spec: {row[2]}")

## Step 2 — Use containment operator (@>) to find parts matching a spec

The `@>` operator asks "does the JSON document on the left contain the JSON sub-document on the right?" This is the workhorse for spec matching. Find parts that match a spec. The SQL:

```sql
SELECT part_id, name, spec
  FROM factory.part
 WHERE spec @> '{"material": "steel"}'
 LIMIT 5
```

In [ ]:
sql = """
SELECT part_id, name, spec
  FROM factory.part
 WHERE spec @> '{"material": "steel"}'
 LIMIT 5
"""
rows = run_sql(sql)
print("Parts with material='steel' (using @> containment):")
for row in rows:
    print(f"  {row[0]}: {row[1]}")
    print(f"    spec: {row[2]}")

## Step 3 — Use path extraction (#>>) and key exists (?) operators

The `#>>` operator extracts a value from a path in the JSON. Use it to extract a field for filtering or display. The `?` operator checks if a key exists. Together, they let you filter and display JSON fields like columns. The SQL:

```sql
SELECT part_id, name, spec #>> '{material}' as material
  FROM factory.part
 WHERE spec ? 'material'
 LIMIT 5
```

In [ ]:
sql = """
SELECT part_id, name, spec #>> '{material}' as material
  FROM factory.part
 WHERE spec ? 'material'
 LIMIT 5
"""
rows = run_sql(sql)
print("Extracted material field from JSONB spec:")
for row in rows:
    print(f"  {row[0]}: material={row[2]}")

## Step 4 — Create a GIN index on JSONB

JSONB queries are fast even without an index. With a GIN (Generalized Inverted Index) on the JSONB column, they are faster. The SQL:

```sql
CREATE INDEX IF NOT EXISTS idx_part_spec_gin ON factory.part
  USING gin(spec jsonb_path_ops)
```

In [ ]:
sql = "CREATE INDEX IF NOT EXISTS idx_part_spec_gin ON factory.part USING gin(spec jsonb_path_ops)"
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)
print("GIN index created on factory.part(spec)")
print("(Queries will use the index if beneficial)")

## Step 5 — Realistic compliance query: parts for harsh environment

Find parts suitable for a harsh environment (must be steel or stainless). The SQL:

```sql
SELECT part_id, name, spec
  FROM factory.part
 WHERE spec @> '{"material": "steel"}'
    OR spec @> '{"material": "stainless"}'
 LIMIT 10
```

In [ ]:
sql = """
SELECT part_id, name, spec
  FROM factory.part
 WHERE spec @> '{"material": "steel"}'
    OR spec @> '{"material": "stainless"}'
 LIMIT 10
"""
rows = run_sql(sql)
print("Parts suitable for harsh environment (steel or stainless):")
for row in rows:
    print(f"  {row[0]}: {row[1]}")
    if row[2]:
        print(f"    {row[2]}")

## What you learned

You mastered JSONB: the containment operator (@>) for semantic matching, path extraction (#>>) and key checks (?) for drill-down, and GIN indexes for speed. Postgres gives you flexibility of schema-less storage with the power of a relational query engine.

Next: **Geospatial queries** use PostGIS to find locations within a distance, nearest neighbors, and spatial indexes.